# Week 3 — Phase A1: Config-Only Pipeline Fix

## Strategy
No code changes. Only YOLO CLI/training hyperparameters.
Train YOLOv11n baseline on NEU-DET with fixed augmentations.

## What Changed vs Week 1-2 Training
| Param | Before | After | Why |
|-------|--------|-------|-----|
| mosaic | 1.0 | **0.0** | 4× upscale from 200→800px blurs fine crazing features |
| mixup | 0.0 | **0.15** | Regularization for small 1200-image dataset |
| degrees | 0.0 | **10.0** | Defects are orientation-invariant |
| translate | 0.1 | **0.2** | More position diversity |
| scale | 0.5 | **0.6** | Moderate size variation |
| shear | 0.0 | **5.0** | Simulates off-angle camera |
| epochs | 300 | **400** | Best epoch was 274 — model needed more time |
| patience | 75 | **100** | More room before early stopping |

## Expected
- **Before:** 75.8% mAP@0.5 (Week 2 baseline v2)
- **Expected:** 78-79% mAP@0.5
- **Train time:** ~5 hours (RTX 2000 Ada, 17GB VRAM)

## Setup

In [2]:
%matplotlib inline
import torch
import json
import sys
from pathlib import Path
from ultralytics import YOLO
import warnings
warnings.filterwarnings("ignore")

# Project root
PROJECT_ROOT = Path(r"D:\DigiSteel-Yolo\DigiSteel-YOLO")
sys.path.insert(0, str(PROJECT_ROOT))

# Paths
DATA_YAML = str(PROJECT_ROOT / "configs" / "data" / "neu_det.yaml")
EVALS_DIR = PROJECT_ROOT / "evals"
EVALS_DIR.mkdir(exist_ok=True)

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({vram_total:.1f} GB VRAM)")

PyTorch 2.6.0+cu124, CUDA: True
GPU: NVIDIA RTX 2000 Ada Generation (17.2 GB VRAM)


## Dataset Verification

In [3]:
# Verify dataset exists and count images
for split in ["train", "val", "test"]:
    img_dir = PROJECT_ROOT / "datasets" / "NEU-DET" / "yolo" / "images" / split
    lbl_dir = PROJECT_ROOT / "datasets" / "NEU-DET" / "yolo" / "labels" / split
    if img_dir.exists() and lbl_dir.exists():
        n_imgs = len(list(img_dir.glob("*")))
        n_lbls = len(list(lbl_dir.glob("*")))
        print(f"  {split:6s}: {n_imgs:4d} images, {n_lbls:4d} labels")
    else:
        print(f"  {split}: MISSING!")

print(f"\nConfig: {DATA_YAML}")
print(f"Ready to train.")

  train : 1290 images, 1290 labels
  val   :  344 images,  344 labels
  test  :  166 images,  166 labels

Config: D:\DigiSteel-Yolo\DigiSteel-YOLO\configs\data\neu_det.yaml
Ready to train.


## GPU Memory Check Before Training

In [4]:
# Quick memory check before committing to training
print("GPU memory before training:")
!nvidia-smi --query-gpu=memory.used,memory.free,memory.total --format=csv

GPU memory before training:
memory.used [MiB], memory.free [MiB], memory.total [MiB]
4508 MiB, 11611 MiB, 16380 MiB


## Train — YOLOv11n Baseline (Fixed Config)

In [5]:
# ---- Week 3 A1: Config-Only Fix ----

# Training hyperparameters (17GB VRAM — can use larger batch)
TRAIN_ARGS = {
    "data": DATA_YAML,
    "epochs": 400,          # was 300 — model needed more time
    "imgsz": 800,           # keep 800
    "batch": 16,            # was 32 — cuDNN OOM at 800px            # 17GB VRAM — batch=32 OOM at 800px, using 16
    "seed": 42,
    "project": str(PROJECT_ROOT / "runs"),
    "name": "week3_a1_baseline",
    "exist_ok": True,
    "patience": 100,        # was 75
    "amp": True,
    "cos_lr": True,
    "close_mosaic": 0,      # no mosaic, no need to close it
    "verbose": True,
    
    # --- Augmentation (the key changes) ---
    "mosaic": 0.0,          # was 1.0 — 4× upscale blurs fine features
    "mixup": 0.15,          # was 0.0  — regularization for small dataset
    "degrees": 10.0,        # was 0.0  — orientation invariance
    "translate": 0.2,       # was 0.1  — more position diversity
    "scale": 0.6,           # was 0.5  — moderate size variation
    "shear": 5.0,           # was 0.0  — off-angle camera simulation
    "fliplr": 0.5,          # keep
    "flipud": 0.0,          # keep off — unnatural for steel direction
    "hsv_h": 0.015,         # keep (low — grayscale images)
    "hsv_s": 0.7,           # keep (low — grayscale images)
    "hsv_v": 0.4,           # keep (low — grayscale images)
    "copy_paste": 0.0,      # keep off — wrong for texture defects
}

print("Training YOLOv11n with fixed augmentations...")
print(f"  mosaic={TRAIN_ARGS['mosaic']}, mixup={TRAIN_ARGS['mixup']}, degrees={TRAIN_ARGS['degrees']}")
print(f"  translate={TRAIN_ARGS['translate']}, scale={TRAIN_ARGS['scale']}, shear={TRAIN_ARGS['shear']}")
print(f"  epochs={TRAIN_ARGS['epochs']}, patience={TRAIN_ARGS['patience']}, batch={TRAIN_ARGS['batch']}")

model = YOLO("yolo11n.pt")  # detection model, NOT classification
results = model.train(**TRAIN_ARGS)

print(f"\nTraining complete!")
print(f"Best weights: {PROJECT_ROOT / 'runs' / 'week3_a1_baseline' / 'weights' / 'best.pt'}")

Training YOLOv11n with fixed augmentations...
  mosaic=0.0, mixup=0.15, degrees=10.0
  translate=0.2, scale=0.6, shear=5.0
  epochs=400, patience=100, batch=16
New https://pypi.org/project/ultralytics/8.4.67 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.253  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\DigiSteel-Yolo\DigiSteel-YOLO\configs\data\neu_det.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=400, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=Fa

## Evaluate

In [6]:
# Fresh evaluation on validation set
weights_path = str(PROJECT_ROOT / "runs" / "week3_a1_baseline" / "weights" / "best.pt")
model = YOLO(weights_path)
metrics = model.val(data=DATA_YAML, split='test', imgsz=800, verbose=True)

# Extract key metrics
results_dict = {
    "phase": "Week3-A1",
    "description": "Config-only fix — mosaic=0, mixup=0.15, degrees=10, translate=0.2, scale=0.6, shear=5, epochs=400",
    "mAP50": round(float(metrics.box.map50), 4),
    "mAP50_95": round(float(metrics.box.map), 4),
    "precision": round(float(metrics.box.mp), 4),
    "recall": round(float(metrics.box.mr), 4),
    "f1": round(2 * float(metrics.box.mp * metrics.box.mr) / (float(metrics.box.mp) + float(metrics.box.mr) + 1e-7), 4),
}

print(f"\n=== Week 3 A1 Results ===")
print(f"mAP@0.5:       {results_dict['mAP50']*100:.1f}%")
print(f"mAP@0.5:0.95:  {results_dict['mAP50_95']*100:.1f}%")
print(f"Precision:     {results_dict['precision']*100:.1f}%")
print(f"Recall:        {results_dict['recall']*100:.1f}%")
print(f"F1:            {results_dict['f1']*100:.1f}%")

Ultralytics 8.3.253  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 386.4147.8 MB/s, size: 18.9 KB)
val: Scanning D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\labels\val.cache... 344 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 344/344  0.0s
val: D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\images\val\crazing_120.jpg: 1 duplicate labels removed
val: D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\images\val\patches_198.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 6.9it/s 3.2s0.1s
                   all        344        819      0.737      0.703      0.748      0.416
               crazing         61        137      0.474      0.414      0.399      0.133
             inclusion         75  

## Per-Class Analysis

In [7]:
# Per-class mAP breakdown (safe API for ultralytics 8.3.x)
class_names = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

try:
    per_class_ap50 = {}
    # Try maps first (safe in 8.3.x), then fall back to ap_class_index + ap50
    if hasattr(metrics.box, 'maps') and metrics.box.maps is not None:
        maps_list = metrics.box.maps.tolist() if hasattr(metrics.box.maps, 'tolist') else list(metrics.box.maps)
        for i, ap in enumerate(maps_list):
            if i < len(class_names):
                per_class_ap50[class_names[i]] = round(float(ap), 4)
    elif hasattr(metrics.box, 'ap_class_index'):
        for i, cls_id in enumerate(metrics.box.ap_class_index):
            if cls_id < len(class_names):
                per_class_ap50[class_names[cls_id]] = round(float(metrics.box.ap50[i]), 4)

    if per_class_ap50:
        print("Per-class mAP@0.5:")
        for name in class_names:
            ap = per_class_ap50.get(name, 0)
            bar = "█" * int(ap * 50)
            print(f"  {name:20s}: {ap*100:5.1f}%  {bar}")
        results_dict["per_class_mAP50"] = per_class_ap50
    else:
        print("Per-class AP not available.")
        print("Run model.val(split='test') separately for per-class breakdown.")
except Exception as e:
    print(f"Per-class AP extraction failed: {e}")
    print("Run model.val(split='test') separately for per-class breakdown.")

Per-class mAP@0.5:
  crazing             :  13.3%  ██████
  inclusion           :  45.2%  ██████████████████████
  patches             :  60.3%  ██████████████████████████████
  pitted_surface      :  49.3%  ████████████████████████
  rolled-in_scale     :  27.5%  █████████████
  scratches           :  54.1%  ███████████████████████████


## Save Results

In [8]:
# Save to evals folder
results_path = EVALS_DIR / "week3_a1_results.json"
with open(results_path, "w") as f:
    json.dump(results_dict, f, indent=2)

print(f"Results saved to: {results_path}")

# Compare with previous week
print(f"\n=== Comparison ===")
print(f"Week 2 Baseline v2:  75.8% mAP@0.5")
print(f"Week 3 A1:          {results_dict['mAP50']*100:.1f}% mAP@0.5")
delta = (results_dict['mAP50'] - 0.758) * 100
print(f"Delta:              {delta:+.1f}%")
print(f"\nDone!")

Results saved to: D:\DigiSteel-Yolo\DigiSteel-YOLO\evals\week3_a1_results.json

=== Comparison ===
Week 2 Baseline v2:  75.8% mAP@0.5
Week 3 A1:          74.8% mAP@0.5
Delta:              -1.0%

Done!
